In [19]:
import os
import json
import pickle
import pathlib
import numpy as np
import scipy.interpolate as interp
from tqdm import tqdm # Used for the loading progress bar

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [20]:
# Set your directory
dir = r"/Volumes/ESSD/BatteryLife" if os.name == 'posix' else r"D:\BatteryLife"

# Function to extract labels
def get_dict(labels_dir):
    dict_labels = {}
    content = os.listdir(labels_dir)
  
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
            continue
        with open(os.path.join(labels_dir, f), 'r') as file:
            data = json.load(file)
            if 'Tongji' in f:
               temp_dict = {}
               for k, v in data.items():
                    k_new = k.replace("#", "-")
                    temp_dict.update({k_new:v})
               data = temp_dict   
            dict_labels.update(data)
            
    k = list(dict_labels.keys())
    v = list(dict_labels.values())
    return k, v

labels_path = os.path.join(dir, "Life labels")
cycles_file, soh = get_dict(labels_path)
print(f"Found {len(soh)} labels.")

Found 1208 labels.


In [21]:

def get_length(dir):
    content = os.listdir(dir)
    length = 0
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
            continue
        with open(os.path.join(dir, f), 'r') as file:
            data = json.load(file)
            length += len(data)
    return length


        

In [22]:
content = os.listdir(dir)
print(f"###### \n ---- Content ---- \n{content} \n######")
folders = [f for f in content if os.path.isdir(os.path.join(dir, f)) and (f != "Life labels" and f != "READMEs") and not f.startswith("._") and not f.startswith(".DS_Store")]
pkl_files = [[] for _ in folders]
for i, folder in enumerate(folders) :
    folder_path = os.path.join(dir, folder)
    for file in os.listdir(folder_path):
        if file.endswith('.pkl'):
            pkl_files[i].append(file)
print(len(pkl_files))

###### 
 ---- Content ---- 
['CALB', 'CALCE', 'HNEI', 'HUST', 'ISU_ILCC', 'Life labels', 'MATR', 'MICH', 'MICH_EXP', 'NA-ion', 'README.md', 'READMEs', 'RWTH', 'SDU', 'SNL', 'Stanford', 'Stanford_2', 'Tongji', 'UL_PUR', 'XJTU', 'ZN-coin', '.DS_Store', '._.DS_Store', '._README.md'] 
######
18


In [23]:
labels = os.path.join(dir, "Life labels")
def get_dict(labels):
    dict_labels = {}
    content = os.listdir(labels)
    length = 0
  
    for f in content:
        if f.startswith("._") or f.startswith(".DS_Store"):
          continue
        with open(os.path.join(labels, f), 'r') as file:
            data = json.load(file)
            if 'Tongji' in f:
               temp_dict = {}
               for k, v in data.items():
                    k_new = k.replace("#", "-")
                    temp_dict.update({k_new:v})
               data = temp_dict   
            dict_labels.update(data)
    k= list(dict_labels.keys())
    v= list(dict_labels.values())
    return k,v


In [24]:
import numpy as np
import scipy.interpolate as interp

def get_cycle_data(data, cycle_format):
    
    arr_total = np.zeros((cycle_format, 3, 300))
    
    for i in range(cycle_format):

        raw_time = np.array(data['cycle_data'][i]['time_in_s'])
        scaled_time = raw_time - raw_time[0]
        raw_current = np.array(data['cycle_data'][i]['current_in_A'])
        raw_voltage = np.array(data['cycle_data'][i]['voltage_in_V'])
        dt = np.diff(scaled_time)
        avg_current = 0.5 * (raw_current[1:] + raw_current[:-1])
        
        cumulative_charge = np.concatenate((
            [0],
            np.cumsum(avg_current * dt)
        ))
        
        capacity_Ah = cumulative_charge / 3600.0

        t_new = np.linspace(0, scaled_time[-1], num=300)
        f_current = interp.interp1d(scaled_time, raw_current, kind='linear')
        f_voltage = interp.interp1d(scaled_time, raw_voltage, kind='linear')
        f_capacity = interp.interp1d(scaled_time, capacity_Ah, kind='linear')
        current_interpolated = f_current(t_new) / data['nominal_capacity_in_Ah']
        voltage_interpolated = f_voltage(t_new) / data['max_voltage_limit_in_V']
        capacity_interpolated = f_capacity(t_new) / data['nominal_capacity_in_Ah']
        
        arr_total[i] = np.stack(
            (current_interpolated,
             voltage_interpolated,
             capacity_interpolated),
            axis=0
        )
    
    return arr_total


In [35]:
# Set up a local folder to save the fast-loading tensors
local_save_dir = "./processed_battery_data" 
os.makedirs(local_save_dir, exist_ok=True)

x_path = os.path.join(local_save_dir, "X_features.pt")
y_path = os.path.join(local_save_dir, "y_labels.pt")

# Check if we already processed and saved the data locally
if os.path.exists(x_path) and os.path.exists(y_path):
    print("Loading pre-processed tensors from local storage...")
    X_tensor = torch.load(x_path)
    y_tensor = torch.load(y_path)
    print("Loaded successfully!")

else:
    print("Tensors not found locally. Extracting from external drive (this will take a few minutes)...")
    
    # Fast path mapping: Find all paths once so we don't have to search later
    file_paths = {}
    for loc in pathlib.Path(dir).rglob('*.pkl'):
        file_paths[loc.name] = loc

    all_features = []
    all_labels = []

    # Loop through all files and extract the features
    for idx in tqdm(range(len(soh))):
        file_name = cycles_file[idx]
        if file_name not in file_paths:
            print(f"Skipping missing file: {file_name}")
            continue
            
        with open(file_paths[file_name], 'rb') as file:
            data = pickle.load(file)

        features = get_cycle_data(data, cycle_format=1)
        all_features.append(features)
        all_labels.append(soh[idx])

    # Convert lists to PyTorch tensors
    X_tensor = torch.tensor(np.array(all_features), dtype=torch.float32)
    y_tensor = torch.tensor(np.array(all_labels), dtype=torch.float32)
    
    # Save them locally for next time!
    torch.save(X_tensor, x_path)
    torch.save(y_tensor, y_path)
    print(f"Saved tensors to {local_save_dir}!")

print(f"Total Features Shape: {X_tensor.shape}")

# --- NORMALIZATION ---
# Find the maximum cycle life in your dataset
y_max = y_tensor.max()
print(f"Max cycle life in dataset: {y_max.item()} cycles")

# Scale all targets to be between 0 and 1
y_tensor_normalized = y_tensor / y_max

# Create the dataset using the NORMALIZED labels
dataset = TensorDataset(X_tensor, y_tensor_normalized)
print(f"Size of dataset: {len(dataset)}")

# Safely split data (1000 for training, the rest for validation)
train_size = 1000
val_size = len(dataset) - train_size
train_data, val_data = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=80, shuffle=True, num_workers=0)
validation_loader = DataLoader(val_data, batch_size=80, shuffle=False, num_workers=0)

Loading pre-processed tensors from local storage...
Loaded successfully!
Total Features Shape: torch.Size([1208, 1, 3, 300])
Max cycle life in dataset: 4999.0 cycles
Size of dataset: 1208


In [33]:
train_features, train_labels = next(iter(train_loader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels}")
print(f"Squeezed shape: {train_features.squeeze(1).shape}")

Feature batch shape: torch.Size([64, 1, 3, 300])
Labels batch shape: tensor([0.1294, 0.1316, 0.1374, 0.0478, 0.0994, 0.1158, 0.1504, 0.0814, 0.0986,
        0.1298, 0.0294, 0.2567, 0.3821, 0.1338, 0.1102, 0.2098, 0.4755, 0.0526,
        0.0234, 0.0304, 0.1014, 0.0730, 0.0870, 0.0738, 0.1120, 0.8568, 0.6169,
        0.0750, 0.1972, 0.0330, 0.1500, 0.3659, 0.0348, 0.0360, 0.1924, 0.0640,
        0.1340, 0.0976, 0.1350, 0.1584, 0.0984, 0.1278, 0.0510, 0.0264, 0.0732,
        0.2637, 0.0464, 0.5805, 0.2012, 0.1748, 0.0426, 0.1652, 0.0940, 0.1584,
        0.0968, 0.0548, 0.0248, 0.0360, 0.1050, 0.6423, 0.1068, 0.1870, 0.1458,
        0.1408])
Squeezed shape: torch.Size([64, 3, 300])


In [37]:
import torch
import torch.nn as nn

# ==========================
# 1. SETUP APPLE SILICON GPU
# ==========================
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Training on device: {device}")

# Move your scaling factor to the GPU so the math doesn't crash later
y_max = y_max.to(device)

# ==========================
# 2. DEFINE LSTM MODEL
# ==========================
class LSTMnetwork(nn.Module):
    def __init__(self, hidden_size=64, num_layers=2):
        super(LSTMnetwork, self).__init__()

        # Upgraded to LSTM, increased hidden size, and added a layer
        self.lstm = nn.LSTM(
            input_size=3, 
            hidden_size=hidden_size, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=0.2 # Helps prevent overfitting when using multiple layers
        )

        self.out = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # Squeeze out the extra dimension and swap axes
        x = x.squeeze(1).permute(0, 2, 1) 
        
        # LSTM returns the output and a tuple of (hidden_state, cell_state)
        _, (hidden_layers, _) = self.lstm(x)
        
        # We take the hidden state from the last layer to make our prediction
        cycle_to_80_pred = self.out(hidden_layers[-1]) 

        return cycle_to_80_pred.flatten()

# ==========================
# 3. INITIALIZE MODEL & OPTIMIZER
# ==========================
# Crucial: Move the model to the Mac GPU
model = LSTMnetwork(hidden_size=64, num_layers=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# ==========================
# 4. TRAINING & VALIDATION LOOP
# ==========================
epochs = 250

# Lists to keep track of our errors so we can plot them later
train_mae_history = []
val_mae_history = []

for epoch in range(epochs):
    
    # --- TRAINING PHASE ---
    model.train() # Set the model to training mode (enables gradients and dropout)
    train_loss = 0
    train_mae = 0

    for batch_x, batch_y in train_loader:
        # Move data to the Mac GPU
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        
        preds = model(batch_x)   
        
        # Calculate MSE loss on normalized data for the optimizer
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()

        # Calculate MAE in real cycle numbers for human readability
        with torch.no_grad():
            preds_real = preds * y_max
            batch_y_real = batch_y * y_max
            mae = torch.abs(preds_real - batch_y_real).mean()
            train_mae += mae.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_mae = train_mae / len(train_loader)
    train_mae_history.append(avg_train_mae)

    # --- VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode (turns off dropout, locks weights)
    val_loss = 0
    val_mae = 0

    # torch.no_grad() tells PyTorch not to calculate gradients (saves memory/time)
    with torch.no_grad():
        for batch_x, batch_y in validation_loader:
            # Move data to the Mac GPU
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            preds = model(batch_x)
            
            # Calculate validation loss
            loss = criterion(preds, batch_y)
            val_loss += loss.item()
            
            # Calculate validation MAE in real cycle numbers
            preds_real = preds * y_max
            batch_y_real = batch_y * y_max
            mae = torch.abs(preds_real - batch_y_real).mean()
            val_mae += mae.item()

    avg_val_loss = val_loss / len(validation_loader)
    avg_val_mae = val_mae / len(validation_loader)
    val_mae_history.append(avg_val_mae)

    # --- PRINT RESULTS ---
    print(f"Epoch {epoch:02d} | "
          f"Train Error: {avg_train_mae:.1f} cycles | "
          f"Val Error: {avg_val_mae:.1f} cycles")

Training on device: mps
Epoch 00 | Train Error: 507.9 cycles | Val Error: 411.4 cycles
Epoch 01 | Train Error: 436.0 cycles | Val Error: 422.1 cycles
Epoch 02 | Train Error: 421.9 cycles | Val Error: 404.8 cycles
Epoch 03 | Train Error: 419.4 cycles | Val Error: 410.6 cycles
Epoch 04 | Train Error: 416.5 cycles | Val Error: 399.2 cycles
Epoch 05 | Train Error: 432.0 cycles | Val Error: 394.0 cycles
Epoch 06 | Train Error: 408.3 cycles | Val Error: 393.5 cycles
Epoch 07 | Train Error: 399.7 cycles | Val Error: 397.9 cycles
Epoch 08 | Train Error: 389.1 cycles | Val Error: 392.3 cycles
Epoch 09 | Train Error: 379.7 cycles | Val Error: 386.5 cycles
Epoch 10 | Train Error: 384.4 cycles | Val Error: 379.2 cycles
Epoch 11 | Train Error: 382.7 cycles | Val Error: 383.6 cycles
Epoch 12 | Train Error: 394.2 cycles | Val Error: 385.3 cycles
Epoch 13 | Train Error: 390.9 cycles | Val Error: 388.5 cycles
Epoch 14 | Train Error: 382.6 cycles | Val Error: 375.4 cycles
Epoch 15 | Train Error: 385.2 c